# Agentic Tool-Calling with Nemotron 3 Super

This notebook demonstrates how to build multi-step AI agents using **Nemotron 3 Super's** structured tool-calling capabilities. We progress from a single tool call to a fully autonomous agent loop that plans, executes, and synthesizes.

| Component | Model | Parameters | Deployment |
|-----------|-------|------------|------------|
| **Reasoning + Tool Calling** | `nvidia/nemotron-3-super-120b-a12b` | 120B total / 12B active | NVIDIA API |

**Prerequisites:** An NVIDIA API key from [build.nvidia.com](https://build.nvidia.com/).

## 1. Setup

In [ ]:
!pip install -q openai

In [ ]:
import json
import os
from getpass import getpass

from openai import OpenAI

NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY") or getpass("NVIDIA API key: ").strip()

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
)

MODEL = "nvidia/nemotron-3-super-120b-a12b"

print(f"Using model: {MODEL}")

## 2. Define Tools

We define a set of tools that our agent can use. Each tool has a JSON schema describing its parameters, following the OpenAI function-calling format that Nemotron 3 Super supports natively.

Our agent will be a **research assistant** that can search for information, read documents, extract structured data, perform calculations, and save reports.

In [ ]:
# Tool definitions using OpenAI-compatible JSON schema format
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_knowledge_base",
            "description": "Search a knowledge base for information on a topic. Returns a list of relevant snippets with source references.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query"
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum number of results to return",
                        "default": 3
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_document",
            "description": "Read the full text of a document given its identifier.",
            "parameters": {
                "type": "object",
                "properties": {
                    "document_id": {
                        "type": "string",
                        "description": "The unique identifier of the document to read"
                    }
                },
                "required": ["document_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "extract_structured_data",
            "description": "Extract structured fields from unstructured text. Returns a JSON object with the requested fields.",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {
                        "type": "string",
                        "description": "The text to extract data from"
                    },
                    "fields": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of field names to extract"
                    }
                },
                "required": ["text", "fields"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression and return the result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A mathematical expression to evaluate (e.g., '(100 * 1.05) - 50')"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "save_report",
            "description": "Save a research report with a title and content.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string",
                        "description": "The report title"
                    },
                    "content": {
                        "type": "string",
                        "description": "The full report content in markdown format"
                    }
                },
                "required": ["title", "content"]
            }
        }
    }
]

print(f"Defined {len(TOOLS)} tools: {[t['function']['name'] for t in TOOLS]}")

### Simulated Tool Implementations

These implementations simulate real tool behavior so the notebook is self-contained and requires no external services beyond the NVIDIA API.

In [ ]:
# Simulated knowledge base for the research assistant
KNOWLEDGE_BASE = {
    "nemotron-architecture": {
        "title": "Nemotron 3 Super Architecture Overview",
        "content": (
            "Nemotron 3 Super is a 120.6B parameter hybrid Mamba-Transformer "
            "Latent Mixture-of-Experts model with 12.7B active parameters per "
            "forward pass. It uses 64 experts with top-4 routing for MLP layers "
            "and Latent MoE attention with 16 experts (top-2 routing). The hybrid "
            "architecture alternates between 32 Mamba-2 layers and 32 Transformer "
            "layers. Context window extends to 1M tokens via YaRN-based positional "
            "interpolation. Training used 30T tokens across pretraining, SFT, and "
            "a three-stage RL pipeline."
        ),
    },
    "nemotron-benchmarks": {
        "title": "Nemotron 3 Super Benchmark Results",
        "content": (
            "PinchBench: 85.6% (best open model). MATH-500: 97.4%. AIME 2025: 72.2%. "
            "GPQA Diamond: 71.1%. LiveCodeBench v6: 63.3%. HumanEval: 92.1%. "
            "SWE-Bench Verified: 55.4%. TerminalBench: 40.6%. TauBench V2 Airline: 62.0%. "
            "The model achieves these scores with only 12B active parameters, "
            "delivering 5x higher throughput than dense models of similar accuracy."
        ),
    },
    "nemotron-training": {
        "title": "Nemotron 3 Super Training Pipeline",
        "content": (
            "Three-stage training: (1) Pretraining on 30T tokens with curriculum "
            "learning across code, math, science, and general text. (2) Multi-domain "
            "SFT over 7M samples covering 15+ data domains including competition "
            "math/code, software engineering, agentic programming, CUDA, financial "
            "reasoning, and more. Uses a novel two-stage SFT loss. (3) Three-stage "
            "RL: multi-environment RLVR across 21 environments and 37 datasets, "
            "SWE-RL with container-isolated sandbox execution, and RLHF with a "
            "principle-following GenRM."
        ),
    },
    "mamba-architecture": {
        "title": "Mamba-2 State Space Model Architecture",
        "content": (
            "Mamba-2 is a selective state space model that achieves linear-time "
            "sequence processing. Unlike attention which scales quadratically, "
            "Mamba-2 maintains a compressed state that is updated incrementally. "
            "This enables efficient processing of very long sequences. In the "
            "hybrid architecture, Mamba layers handle sequential dependencies "
            "while Transformer layers provide precise attention for complex "
            "reasoning tasks. The alternating pattern (32 Mamba + 32 Transformer) "
            "balances efficiency and accuracy."
        ),
    },
    "latent-moe": {
        "title": "Latent Mixture-of-Experts Explained",
        "content": (
            "Latent MoE is an architectural innovation that applies the MoE "
            "pattern to attention layers, not just MLP layers. Traditional MoE "
            "routes tokens to different MLP experts. Latent MoE extends this by "
            "routing to different attention heads, effectively allowing 4x more "
            "experts at the same compute cost. Nemotron 3 Super uses 16 Latent "
            "MoE attention experts with top-2 routing in each Transformer layer, "
            "alongside 64 MLP experts with top-4 routing."
        ),
    },
}

SAVED_REPORTS = []


def execute_tool(name: str, arguments: dict) -> str:
    """Execute a tool call and return the result as a string."""
    if name == "search_knowledge_base":
        query = arguments["query"].lower()
        max_results = arguments.get("max_results", 3)
        results = []
        for doc_id, doc in KNOWLEDGE_BASE.items():
            # Simple keyword matching for simulation
            if any(word in doc["content"].lower() or word in doc["title"].lower()
                   for word in query.split()):
                results.append({
                    "document_id": doc_id,
                    "title": doc["title"],
                    "snippet": doc["content"][:150] + "...",
                })
        return json.dumps(results[:max_results])

    elif name == "read_document":
        doc_id = arguments["document_id"]
        if doc_id in KNOWLEDGE_BASE:
            doc = KNOWLEDGE_BASE[doc_id]
            return json.dumps({"title": doc["title"], "content": doc["content"]})
        return json.dumps({"error": f"Document '{doc_id}' not found"})

    elif name == "extract_structured_data":
        text = arguments["text"]
        fields = arguments["fields"]
        # Simulate extraction by returning placeholder values
        extracted = {}
        for field in fields:
            field_lower = field.lower()
            if "parameter" in field_lower and "120" in text:
                extracted[field] = "120.6B total, 12.7B active"
            elif "active" in field_lower and "12" in text:
                extracted[field] = "12.7B"
            elif "context" in field_lower and "1M" in text:
                extracted[field] = "1M tokens"
            elif "expert" in field_lower and "64" in text:
                extracted[field] = "64 MLP experts (top-4), 16 attention experts (top-2)"
            elif "benchmark" in field_lower or "score" in field_lower:
                extracted[field] = "PinchBench 85.6%, MATH-500 97.4%"
            else:
                extracted[field] = f"[extracted from text for '{field}']"
        return json.dumps(extracted)

    elif name == "calculate":
        expression = arguments["expression"]
        try:
            # Only allow safe mathematical expressions
            allowed = set("0123456789+-*/().% ")
            if all(c in allowed for c in expression):
                result = eval(expression)  # noqa: S307
                return json.dumps({"expression": expression, "result": result})
            return json.dumps({"error": "Invalid expression"})
        except Exception as e:
            return json.dumps({"error": str(e)})

    elif name == "save_report":
        title = arguments["title"]
        content = arguments["content"]
        report = {"title": title, "content": content, "id": len(SAVED_REPORTS) + 1}
        SAVED_REPORTS.append(report)
        return json.dumps({"status": "saved", "report_id": report["id"], "title": title})

    return json.dumps({"error": f"Unknown tool: {name}"})


print("Tool implementations ready.")

## 3. Single Tool Call

The simplest case: the model decides to call one tool to answer a question.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a research assistant. Use the provided tools to answer questions accurately."},
        {"role": "user", "content": "What is Nemotron 3 Super's score on PinchBench?"},
    ],
    tools=TOOLS,
    tool_choice="auto",
)

message = response.choices[0].message
print(f"Model decided to call: {message.tool_calls[0].function.name}")
print(f"With arguments: {message.tool_calls[0].function.arguments}")

# Execute the tool
tool_call = message.tool_calls[0]
result = execute_tool(tool_call.function.name, json.loads(tool_call.function.arguments))
print(f"\nTool result: {result}")

Now feed the tool result back to the model so it can formulate a natural language answer:

In [ ]:
# Continue the conversation with the tool result
follow_up = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a research assistant. Use the provided tools to answer questions accurately."},
        {"role": "user", "content": "What is Nemotron 3 Super's score on PinchBench?"},
        message,  # The assistant's tool-call message
        {"role": "tool", "tool_call_id": tool_call.id, "content": result},
    ],
    tools=TOOLS,
    tool_choice="auto",
)

print("Model's answer:")
print(follow_up.choices[0].message.content)

## 4. Multi-Turn Tool Calling

In a multi-turn scenario, the model calls a tool, gets the result, then decides whether to call another tool or provide a final answer. This enables the model to gather information incrementally.

Here we implement a helper that runs the tool-calling loop until the model produces a final text response.

In [ ]:
def run_tool_loop(
    messages: list[dict],
    tools: list[dict],
    max_turns: int = 10,
    verbose: bool = True,
) -> str:
    """
    Run a tool-calling loop until the model produces a final text response
    or the maximum number of turns is reached.

    Returns the model's final text response.
    """
    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )

        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        # If the model didn't call any tools, we have our final answer
        if not assistant_message.tool_calls:
            if verbose:
                print(f"\n[Turn {turn + 1}] Final answer (no more tool calls)")
            return assistant_message.content

        # Execute each tool call and add results to the conversation
        for tool_call in assistant_message.tool_calls:
            args = json.loads(tool_call.function.arguments)
            if verbose:
                print(f"[Turn {turn + 1}] Calling {tool_call.function.name}({json.dumps(args, indent=2)})")

            result = execute_tool(tool_call.function.name, args)
            if verbose:
                print(f"  -> Result: {result[:200]}{'...' if len(result) > 200 else ''}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            })

    return "[Max turns reached without a final answer]"


print("Tool loop helper defined.")

In [ ]:
# Multi-turn example: a question that requires multiple tool calls
messages = [
    {
        "role": "system",
        "content": (
            "You are a research assistant with access to a technical knowledge base. "
            "Search for information, read documents, and extract data to answer questions. "
            "Always cite your sources by document ID."
        ),
    },
    {
        "role": "user",
        "content": (
            "Compare the Mamba and Transformer components in Nemotron 3 Super. "
            "How many layers of each type are there, and what role does each play?"
        ),
    },
]

answer = run_tool_loop(messages, TOOLS)
print("\n" + "=" * 60)
print("FINAL ANSWER:")
print("=" * 60)
print(answer)

## 5. Autonomous Agent Loop

Now we build a fully autonomous agent. Given a complex research task, the model:
1. **Plans** what information it needs
2. **Searches** the knowledge base
3. **Reads** relevant documents in full
4. **Extracts** structured data
5. **Synthesizes** findings into a report
6. **Saves** the report

The system prompt instructs the model to work autonomously through all these steps.

In [ ]:
AGENT_SYSTEM_PROMPT = """\
You are an autonomous research agent. When given a research task, you must:

1. Search the knowledge base to find relevant documents
2. Read the full text of the most relevant documents
3. Extract structured data from the documents as needed
4. Perform any calculations required
5. Synthesize your findings into a comprehensive report
6. Save the report using the save_report tool

Work through these steps autonomously. Do not ask the user for clarification
- use your best judgment. After saving the report, provide a brief summary
of your findings to the user.

Always be thorough: read multiple documents when available, cross-reference
information, and note any gaps in the available data."""

# Reset saved reports
SAVED_REPORTS.clear()

messages = [
    {"role": "system", "content": AGENT_SYSTEM_PROMPT},
    {
        "role": "user",
        "content": (
            "Research the Nemotron 3 Super model architecture and training pipeline. "
            "I need a report covering: (1) the hybrid architecture design and why it "
            "was chosen, (2) the key training stages and techniques used, (3) the "
            "resulting benchmark performance. Calculate the ratio of active to total "
            "parameters and explain what this means for inference efficiency. "
            "Save a complete report when done."
        ),
    },
]

print("Starting autonomous agent...")
print("=" * 60)
answer = run_tool_loop(messages, TOOLS, max_turns=15)
print("\n" + "=" * 60)
print("AGENT SUMMARY:")
print("=" * 60)
print(answer)

In [ ]:
# View the saved report
if SAVED_REPORTS:
    report = SAVED_REPORTS[-1]
    print(f"Report #{report['id']}: {report['title']}")
    print("-" * 60)
    print(report["content"])
else:
    print("No reports saved yet.")

## 6. Reasoning Modes with Tool Calling

Nemotron 3 Super supports three reasoning modes that affect how it approaches tool-calling tasks:

| Mode | Behavior | Best For |
|------|----------|----------|
| **`reasoning-off`** | Direct tool selection, no internal deliberation | Simple lookups, high-throughput pipelines |
| **`regular`** | Full chain-of-thought before tool selection | Complex multi-step tasks |
| **`low-effort`** | Brief reasoning, then tool selection | Balanced speed/accuracy |

Let's compare how each mode handles the same task.

In [ ]:
REASONING_TASK = (
    "Find out how many MLP experts and attention experts Nemotron 3 Super uses, "
    "then calculate the total number of experts across both types."
)


def run_with_reasoning_mode(mode: str) -> None:
    """Run the same task with a specific reasoning mode."""
    print(f"\n{'=' * 60}")
    print(f"Mode: {mode}")
    print("=" * 60)

    extra_body = {"chat_template_kwargs": {}}
    if mode == "reasoning-off":
        extra_body["chat_template_kwargs"]["enable_thinking"] = False
    elif mode == "regular":
        extra_body["chat_template_kwargs"]["enable_thinking"] = True
    elif mode == "low-effort":
        extra_body["chat_template_kwargs"]["enable_thinking"] = True
        extra_body["chat_template_kwargs"]["low_effort"] = True

    messages = [
        {"role": "system", "content": "You are a research assistant. Use tools to answer accurately."},
        {"role": "user", "content": REASONING_TASK},
    ]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=TOOLS,
        tool_choice="auto",
        extra_body=extra_body,
    )

    msg = response.choices[0].message
    usage = response.usage
    print(f"Tokens used: {usage.total_tokens} (prompt: {usage.prompt_tokens}, completion: {usage.completion_tokens})")

    if msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  Tool call: {tc.function.name}({tc.function.arguments})")
    else:
        print(f"  Direct answer: {msg.content[:200]}")


for mode in ["reasoning-off", "low-effort", "regular"]:
    run_with_reasoning_mode(mode)

## 7. Best Practices

### System Prompt Design

Effective system prompts for tool-calling agents should:
- Clearly state the agent's role and available capabilities
- Specify when to use tools vs. answer directly
- Define the expected workflow (search -> read -> extract -> synthesize)
- Set autonomy level (ask for clarification vs. use best judgment)

### Tool Schema Design

- Use clear, descriptive names (`search_knowledge_base` not `search`)
- Write detailed descriptions - the model reads these to decide which tool to use
- Mark required vs. optional parameters explicitly
- Include `default` values for optional parameters
- Use specific types (`integer` not `number` when appropriate)

### Handling Edge Cases

- **Boolean parameters**: Use JSON `true`/`false`, not Python strings `"True"`/`"False"` (see [issue #52](https://github.com/NVIDIA-NeMo/Nemotron/issues/52))
- **Token limits**: For long tool results, consider truncating or summarizing before passing back
- **Error handling**: Return structured error JSON from tools so the model can recover
- **Max turns**: Always set a maximum iteration count to prevent infinite loops

### Choosing a Reasoning Mode

| Scenario | Recommended Mode |
|----------|------------------|
| Simple data lookup | `reasoning-off` (fastest) |
| Multi-step research task | `regular` (most thorough) |
| Production pipeline with latency constraints | `low-effort` (balanced) |
| Debugging tool-calling behavior | `regular` (shows reasoning) |

## Next Steps

- **Self-hosted deployment**: Replace the NVIDIA API with a local vLLM server for full control. See the [vLLM cookbook](../../usage-cookbook/Nemotron-3-Super/vllm_cookbook.ipynb).
- **Real tools**: Connect to live APIs (web search, databases, file systems) instead of simulated tools.
- **Multi-agent patterns**: Orchestrate multiple Nemotron agents with different system prompts and tool sets.
- **Streaming**: Use `stream=True` with `delta.tool_calls` for real-time tool-call streaming. See the [Getting Started Guide](../Nemotron-3-Super-Getting-Started-Guide/Nemotron-3-Super-Getting-Started-Guide.ipynb).